# process tsv gene expression matrix for analysis
1. scanpy clustering
2. scanpy 3D umap

In [1]:
import sys
sys.path.insert(0, "/home/ubuntu/xenaConvert/")
from xenaConvert import *
import scanpy as sc

In [13]:
input = "exprMat.tsv"
max_fraction = 0.2 # for xenium data, default in scanpy is 0.05
#resolution = 0.2 # Higher resolution values lead to more clusters
resolution = 0.5 # Higher resolution values lead to more clusters
study="16-080L CosMx"
outputdir = "xena"

In [3]:
adata =sc.read_text(input, delimiter="\t")
adata

AnnData object with n_obs × n_vars = 978 × 9955

In [4]:
adata = adata.T

In [5]:
adata, adata.X

(AnnData object with n_obs × n_vars = 9955 × 978,
 array([[0., 0., 0., ..., 0., 0., 0.],
        [1., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 1.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]], dtype=float32))

In [ ]:
adata = basic_analysis(adata, max_fraction = max_fraction, resolution= resolution)

In [8]:
adata

AnnData object with n_obs × n_vars = 9955 × 978
    obs: 'n_genes', 'n_counts', 'louvain', 'leiden'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'log1p', 'hvg', 'pca', 'neighbors', 'louvain', 'leiden', 'louvain_DE', 'leiden_DE'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'distances', 'connectivities'

In [9]:
adata.var.highly_variable[adata.var.highly_variable==True]

ACVR1B    True
ADGRF3    True
ADGRL2    True
ADIPOQ    True
AHI1      True
          ... 
VWF       True
WIF1      True
WNT11     True
WNT5B     True
WNT7A     True
Name: highly_variable, Length: 206, dtype: bool

In [14]:
sc.tl.louvain(adata, resolution = resolution)
sc.tl.leiden(adata, resolution = resolution)

In [15]:
len(set(adata.obs.leiden)), len(set(adata.obs.louvain))

(13, 9)

In [16]:
# DE
sc.tl.rank_genes_groups(adata, groupby="louvain", key_added = "louvain_DE", mask_vars=adata.var.highly_variable, method="wilcoxon")
sc.tl.rank_genes_groups(adata, groupby="leiden", key_added = "leiden_DE", mask_vars=adata.var.highly_variable, method="wilcoxon")

In [17]:
adataToCluster(adata, outputdir, study)

In [12]:
adataToMap(adata, outputdir, study)

unrecognized or ignored map: X_pca
